<a href="https://colab.research.google.com/github/Daniel-534/ModelamientoNCuerpos/blob/main/S_Stars_Dynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evolución dinámica e interacciones gravitacionales del cúmulo de estrellas S en el entorno del agujero negro supermasivo Sgr A*

# Modelado N cuerpos, Mecánica Celeste

## Fuentes

### Artículos

* [An Update on Monitoring Stellar Orbits in the Galactic Center](https://arxiv.org/abs/1611.09144)
* [Kinematic Structure of the Galactic Center S-cluster](https://arxiv.org/abs/2006.11399)
* [Sagittarius A* - The Milky Way Supermassive Black Hole](https://arxiv.org/abs/2503.20081)


### Bases de datos

* [25yrs monitoring of stellar orbits in the GC (Gillessen+, 2017)](https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/ApJ/837/30)
* [Kinematic structure of the Galactic Center S cluster (Ali+, 2020)](https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/ApJ/896/100)

Preguntas de investigación principales:

Dinámica Newtoniana base: ¿Cómo evolucionan las órbitas de las estrellas S (comenzando con un subconjunto de 5 y escalando a 49) al integrar numéricamente sus estados cartesianos iniciales bajo la influencia dominante de Sgr A*?
Perturbaciones de N-cuerpos: ¿Qué tan significativas son las interacciones gravitacionales estrella-estrella en comparación con la atracción central de Sgr A* a lo largo del tiempo? (Comparación entre
N
 simulaciones de 2 cuerpos vs. 1 simulación de
N
+
1
 cuerpos).

In [1]:
!pip install -Uq pymcel celluloid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 37.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.2 MB/s eta 0:00:00


In [2]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
from celluloid import Camera
import pandas as pd

Bienvenido a PyMCel v0.9.18 ¡al infinito y más allá!


## Condiciones Iniciales

La base de datos a emplear es un archivo .tsv que contiene información, como los elementos orbitales, de 39 estrellas delcúmulo estelar cercano a Sagitario A* (Sgr A*), el agujero negro supermasivo central de la Vía Láctea, consta de estrellas rápidas llamadas "estrellas S".

In [10]:
url = "https://raw.githubusercontent.com/Daniel-534/ModelamientoNCuerpos/main/Sgrestrellas.tsv"

df = pd.read_csv(url, sep=';')

df.head()

,Disk,Name,a,e_a,e,e_e,i,e_i,omega,e_omega,Omega,e_Omega,Tclos,q_Tclos,e_Tclos,Simbad,_RA,_DE
0,black,S1,22.675,0.257,0.665,0.003,121.066,0.401,109.893,0.458,352.484,0.286,2000.261,,0.001,Simbad,266.41684,-29.00787
1,black,S2,5.034,0.001,0.887,0.002,137.514,0.401,73.416,0.745,235.634,1.031,2002.390,,0.020,Simbad,266.41685,-29.00777
2,black,S8,16.637,0.182,0.768,0.022,75.057,0.573,337.931,2.120,317.075,0.630,1979.216,,0.037,Simbad,266.41696,-29.00788
3,black,S9,11.125,0.030,0.791,0.036,81.876,0.458,137.854,0.573,158.079,0.229,1972.924,,0.023,Simbad,266.41690,-29.00790
4,black,S12,11.962,0.105,0.906,0.003,33.060,0.516,311.173,0.802,236.173,1.146,1995.881,,0.001,Simbad,266.41682,-29.00772


In [4]:
# Extraemos las primeras 6 estrellas
primeras_6 = df.iloc[:6]

primeras_6

,Disk,Name,a,e_a,e,e_e,i,e_i,omega,e_omega,Omega,e_Omega,Tclos,q_Tclos,e_Tclos,Simbad,_RA,_DE
0,black,S1,22.675,0.257,0.665,0.003,121.066,0.401,109.893,0.458,352.484,0.286,2000.261,,0.001,Simbad,266.41684,-29.00787
1,black,S2,5.034,0.001,0.887,0.002,137.514,0.401,73.416,0.745,235.634,1.031,2002.390,,0.020,Simbad,266.41685,-29.00777
2,black,S8,16.637,0.182,0.768,0.022,75.057,0.573,337.931,2.120,317.075,0.630,1979.216,,0.037,Simbad,266.41696,-29.00788
3,black,S9,11.125,0.030,0.791,0.036,81.876,0.458,137.854,0.573,158.079,0.229,1972.924,,0.023,Simbad,266.41690,-29.00790
4,black,S12,11.962,0.105,0.906,0.003,33.060,0.516,311.173,0.802,236.173,1.146,1995.881,,0.001,Simbad,266.41682,-29.00772
5,black,S13,9.580,1.264,0.415,0.030,24.694,7.219,256.513,11.459,47.842,15.126,2004.015,,0.507,Simbad,266.41679,-29.00781


Convertimos los elementos orbitales clásicos a estado cartesiano:

In [5]:
mu = 1.0

def elementos_a_estado(mu, elementos):
    """
    Convierte elementos orbitales clásicos a estado cartesiano.
    elementos = (p, e, i, W, w, f)
    """

    p, e, i, W, w, f = elementos

    # Momento angular específico
    h = np.sqrt(mu * p)

    # Distancia radial
    r = p / (1 + e * np.cos(f))

    # Posición
    x = r * (np.cos(W)*np.cos(w+f) - np.cos(i)*np.sin(W)*np.sin(w+f))
    y = r * (np.sin(W)*np.cos(w+f) + np.cos(i)*np.cos(W)*np.sin(w+f))
    z = r * (np.sin(i)*np.sin(w+f))

    # Velocidad
    muh = mu / h

    vx = muh * (
        -np.cos(W)*np.sin(w+f)
        - np.cos(i)*np.sin(W)*np.cos(w+f)
        - e*(np.cos(W)*np.sin(w) + np.cos(i)*np.sin(W)*np.cos(w))
    )

    vy = muh * (
        -np.sin(W)*np.sin(w+f)
        + np.cos(i)*np.cos(W)*np.cos(w+f)
        - e*(np.sin(W)*np.sin(w) - np.cos(i)*np.cos(W)*np.cos(w))
    )

    vz = muh * (
        np.sin(i)*np.cos(w+f)
        + e*np.sin(i)*np.cos(w)
    )

    return np.array([x, y, z, vx, vy, vz])

In [6]:
estados = []

for _, estrella in primeras_6.iterrows():

    a = estrella['a']
    e = estrella['e']
    p = a*(1 - e**2)
    i = np.radians(estrella['i'])
    W = np.radians(estrella['Omega'])
    w = np.radians(estrella['omega'])

    f = 0.0   # periastro ?

    elementos = [p, e, i, W, w, f]

    estado = elementos_a_estado(mu, elementos)

    estados.append(estado)

In [7]:
columnas_estado = ['x','y','z','vx','vy','vz']

df_estados = pd.DataFrame(estados, columns=columnas_estado)

df_estados

,x,y,z,vx,vy,vz
0,-3.044613,-3.316143,6.118386,-0.425707,0.139084,-0.136456
1,-0.423510,0.092922,0.368220,0.668894,1.657277,0.351110
2,2.364559,-2.709894,-1.401167,0.296348,-0.054754,0.606001
3,1.516981,-0.848134,1.544553,0.580678,-0.134553,-0.644197
4,-1.001365,-0.220053,-0.461718,0.051191,-1.214017,0.467574
5,2.793172,-4.292175,-2.276752,0.406880,0.290751,-0.048959
